# Problem presentation
In this example, we try to predict the energy consumption used for **all the different sectors** in the NO-5 region of Norway. To do so, we will use different weather variables from the Florida station in Bergen: maximum temperature, maximum wind and precipitation.


![alt text](../../extras/map_of_norge_areas.png "Optional Title")

(photos credits: astrom.no)


# Load python modules

In [ ]:
# Magic formulae so that changes in the .py files are loaded in here
%load_ext autoreload
%autoreload 2
###
    
import numpy as np
import matplotlib.pyplot as plt
import matplotlib 
from tqdm.notebook import tqdm
import copy

import pandas as pd
import xarray as xr

import torch.nn as nn
import torch.optim as optim
import random
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import sklearn


# STEP 1 PREPROCESSING DATA
The first step of the machine learning pipeline is to load the data from a file. Here, our data is stored in the `dataset_energy.csv`. We use the `pandas` library to load the data.

In [ ]:
df = pd.read_csv('data/dataset_energy.csv', index_col=0)
df

Our dataset contains hourly values from 2020 to 2025.

Once the data is loaded, we need to turn the datasets into an ML friendly array. To do this, we choose the variables necessary for the analysis and then we transform them into numpy arrays.

In [ ]:
X = df[['Pr','Tmax','Tmin','v_mean','v_max']].values # X is the array containing the data we use to predict the result.
y = df[['Consumption Cabins and holiday properties',
        'Consumption Household',
        'Consumption Primary Industry',
        'Consumption Secondary Industry',
        'Consumption Tertiary Service',
        'Consumption Total']].values # y is the array representing the values we want to predict.

## Create train, validation and test datasets from `X` and `y`.

In [ ]:
X_train, X_valtest, y_train, y_valtest = sklearn.model_selection.train_test_split(X,y, train_size=0.7)
X_val, X_test, y_val, y_test = sklearn.model_selection.train_test_split(X_valtest,y_valtest, train_size=0.7)

## Data preprocessing
As a default for the project, the data is not tranformed into a friendly format for the neural network. You can change this by applying some sklearn transform to both X and y.

In [ ]:
transform_x = sklearn.preprocessing.FunctionTransformer() # Does nothing for now
transform_y = sklearn.preprocessing.FunctionTransformer() # Does nothing for now
transform_x.fit(X_train)
transform_y.fit(y_train)

X_train = transform_x.transform(X_train)
X_val = transform_x.transform(X_val)
X_test = transform_x.transform(X_test)

y_train = transform_y.transform(y_train)
y_val = transform_y.transform(y_val)
y_test = transform_y.transform(y_test)

> **_CAREFUL:_**  **Pytorch does not work with numpy arrays**.
> 
> `X` and `y` datasets must be converted to tensor, which is a kind of array with additional properties.

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float)
X_val = torch.tensor(X_val, dtype=torch.float)
X_test = torch.tensor(X_test, dtype=torch.float)

y_train = torch.tensor(y_train, dtype=torch.float)
y_val = torch.tensor(y_val, dtype=torch.float)
y_test = torch.tensor(y_test, dtype=torch.float)

> In addition, we will want to use a GPU, which is a specialized computing unit which allows for faster training.
>
> To use the GPU, all data and models must **be** on the GPU. We send data around by using data.to(device).
> 
> We use a device called `cuda`, which corresponds to GPU built by the NVIDIA brand.

In [ ]:
device = 'cuda'

X_train = X_train.to(device)
X_val = X_val.to(device)
X_test = X_test.to(device)

y_train = y_train.to(device)
y_val = y_val.to(device)
y_test = y_test.to(device)

# STEP 2: Define ML model and fit it to the data.


The model we will use in in the `src/model_definition.py` source file. This is where you can change the model.

In [ ]:
from src.define_model import MultiLayerPerceptron3layers

In [ ]:
n_features = 5
n_predictions = 6
model = MultiLayerPerceptron3layers(number_input_features=n_features, number_of_predictions=n_predictions)
model = model.to(device)

model

## Define hyperparameters for training

In [ ]:

# training hyperparameters
n_epochs = 100  # number of epochs to run
batch_size = 256  # size of each batch
learning_rate = 0.001  # learning rate
index_where_batches_start = torch.arange(0, len(X_train), batch_size)

loss_fn = nn.MSELoss()  # mean square error
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Train network
We load training functions from the `src/training.py` file.

In [ ]:
from src.training import training_one_batch, evaluation_one_epoch

In [ ]:
# We keep the weights and biases of the best model
best_loss_value = np.inf   # initialise to infinity
best_weights = None
loss_history = []

# training loop
for epoch in tqdm(range(n_epochs)):
    model.train()
    for start in index_where_batches_start:
    # Select the batch of data
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        training_one_batch(model, X_batch, y_batch, optimizer, loss_fn)
    model.eval()
    best_loss_value, best_weights = evaluation_one_epoch(model, X_val, y_val, 
                                                         loss_fn, loss_history, 
                                                         best_loss_value, best_weights)

# restore model weights to the best model
model.load_state_dict(best_weights)
model = model.eval()


###  Plot training history: verify that the loss function goes down

In [ ]:
with torch.no_grad():
    best_pred = model(X_val).detach().cpu()


fix, ax = plt.subplots()
# ax.set_ylim(0,None)
print("MSE: %.2f" % best_loss_value)
print("RMSE: %.2f" % np.sqrt(best_loss_value))
ax.plot(loss_history)
ax.set_ylabel('Loss function (MSE)')
ax.set_xlabel('Epoch')
ax.set_xlim(0,None)
ax.grid()
plt.show()

# STEP 3: Make predictions and evaluate the model skill
Once the model is **trained** with the data, we can use it to make predictions.

## Step 3.1: Predict `y_val` from `X_val`
We predict the values of `y` from `X` based on the relation learned by the model. For neural networks with pytorch, there is no `model.predict` method. We only use `model(X)`

In [ ]:
y_val_predictions = model(X_val)

## Step 3.2: Plot the results and evaluate the model.
To visually evaluate the model, we plot the predicted `y` values against the true `y` values.


> This part is a bit confusing: pytorch tensors can not be used directly for plots.
>
> Remember how we moved the data to the GPU in the first place? We have to move it back to the "normal" computer unit, the cpu.
> This is done by doing `data.to('cpu')`.
>
> Tensors also have additional properties that can only be stored on GPU and that differentiate them from normal numpy arrays. We also have to remove these additional properties before moving data back to the cpu by doing `data.detach()`.
>
> Overall, we need to write `data_cpu = data.detach().to('cpu')`
>
> Note that we also have to put the y_val back to the 'cpu'.

In [ ]:
y_val_predictions_cpu = y_val_predictions.detach().to('cpu')
y_val_cpu = y_val.detach().to('cpu')

We also transform back the `y` values in the original data distribution

In [ ]:
y_val_predictions_cpu = transform_y.inverse_transform(y_val_predictions_cpu)
y_val_cpu = transform_y.inverse_transform(y_val_cpu)

### Plot the predictions against the true data for the validation

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for k in range(6):
    ax.scatter(y_val_cpu[:,k],y_val_predictions_cpu[:,k], s=5, label=df.columns[5+k])
ax.axline((y_val_cpu.mean(), y_val_cpu.mean()), slope=1, ls='--', color='tab:red')
ax.set_ylabel('Predictions')
ax.set_xlabel('True values')
ax.legend()

ax.grid()

## We can evaluate the model with scores
### For the full dataset and for predicted variables separately

Note that the R2 score can in fact be negative if the prediction is worse than just saying the average value.

In [ ]:
scores = dict()
# Full dataset
rmse = sklearn.metrics.root_mean_squared_error(y_val_cpu.flatten(), y_val_predictions_cpu.flatten())
r2_score = sklearn.metrics.r2_score(y_val_cpu.flatten(), y_val_predictions_cpu.flatten())
scores['All variables together'] = [rmse,r2_score]

for k in range(6):
    rmse = sklearn.metrics.root_mean_squared_error(y_val_cpu[:,k], y_val_predictions_cpu[:,k])
    r2_score = sklearn.metrics.r2_score(y_val_cpu[:,k], y_val_predictions_cpu[:,k])
    scores[df.columns[5+k]] = [rmse,r2_score]
df_scores = pd.DataFrame(scores, index=['RMSE','R2_score']).T

fig, axs = plt.subplots(1,2, sharey=True)
df_scores.RMSE.plot.barh(ax=axs[0], zorder=10)
df_scores.R2_score.plot.barh(ax=axs[1], zorder=10, color='tab:red')
for ax in axs:
    ax.grid(zorder=0)
axs[0].set_title('RMSE')
axs[1].set_title('R2 score')

## Now you're ready to start your projects! 

Now that you've seen the model in action, have a look at how the model is defined in 
`src/define_model.py`, and how it is trained and evaluated in `src/train.py`.

Here are some things you can experiment with during the projects:
- The choice of optimiser
- The batch size
- The learning rate
- The preprocessing applied to the data
- The network definition